# FraudShield AI: Exploratory Data Analysis (EDA)

This notebook analyzes the synthetic enterprise transactions dataset to uncover fraud patterns and prepare the data for machine learning.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import json

# Set style
sns.set_theme(style="whitegrid")
plt.rcParams['figure.figsize'] = (10, 6)

## 1. Dataset Overview

In [ ]:
df = pd.read_csv('../data/synthetic_transactions.csv')
print(f"Dataset Shape: {df.shape}")
df.head()

In [ ]:
print("Missing Values:")
print(df.isnull().sum())
print("\nData Types:")
print(df.dtypes)

## 2. Fraud Distribution (Class Imbalance)

In [ ]:
plt.figure(figsize=(6,4))
ax = sns.countplot(data=df, x='is_fraud', palette='pastel')
plt.title('Fraud vs Non-Fraud Transactions')
plt.xlabel('Is Fraud (0 = No, 1 = Yes)')
plt.ylabel('Count')
for p in ax.patches:
    ax.annotate(f'{p.get_height()}', (p.get_x() + p.get_width() / 2., p.get_height()), ha='center', va='center', xytext=(0, 5), textcoords='offset points')
plt.show()

## 3. Fraud by Feature

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(15, 5))

sns.boxplot(data=df, x='is_fraud', y='amount', ax=axes[0], palette='Set2')
axes[0].set_title('Transaction Amount by Fraud Status')
axes[0].set_yscale('log')

sns.countplot(data=df, x='payment_method', hue='is_fraud', ax=axes[1], palette='Set1')
axes[1].set_title('Fraud by Payment Method')
axes[1].tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(12,5))
sns.countplot(data=df, x='transaction_hour', hue='is_fraud', palette='coolwarm')
plt.title('Fraud by Transaction Hour')
plt.show()

## 4. Correlation Analysis

In [ ]:
numeric_df = df.select_dtypes(include=[np.number])
corr = numeric_df.corr()

plt.figure(figsize=(8,6))
sns.heatmap(corr, annot=True, cmap='coolwarm', fmt=".2f")
plt.title('Correlation Matrix')
plt.show()

## 5. Model Evaluation Review

We evaluate 4 models: Logistic Regression, Decision Tree, Random Forest, and XGBoost.
Based on the output of `train_model.py`:

In [ ]:
import os
metrics_path = '../models/model_metrics.json'
if os.path.exists(metrics_path):
    with open(metrics_path, 'r') as f:
        metrics = json.load(f)
    
    print(f"Champion Model: {metrics['best_model']}\n")
    
    results_df = pd.DataFrame(metrics['metrics']).T
    display(results_df)
    
    results_df[['ROC-AUC', 'F1-Score', 'Precision', 'Recall']].plot(kind='bar', figsize=(10,6), colormap='viridis')
    plt.title('Model Comparison')
    plt.ylabel('Score')
    plt.ylim(0, 1.1)
    plt.xticks(rotation=0)
    plt.legend(loc='lower right')
    plt.show()
else:
    print("Metrics file not found. Run train_model.py first.")